In [0]:
duplicates = spark.sql("""
  SELECT member_id, COUNT(*) AS total
  FROM finance_project.customers_external
  GROUP BY member_id
  HAVING COUNT(*) > 1
  ORDER BY total DESC
""")

In [0]:

duplicates.show()

In [0]:
spark.sql("""
  SELECT member_id, home_ownership, annual_income, grade, sub_grade, tot_hi_cred_limit
  FROM finance_project.customers_external 
  WHERE member_id LIKE 'e4c167053d5418230%'
""").show(10, False)

In [0]:
duplicates2 = spark.sql("""
  SELECT member_id, COUNT(*) AS total
  FROM finance_project.loans_defaulters_delinq_external
  GROUP BY member_id
  HAVING COUNT(*) > 1
  ORDER BY total DESC
""")

In [0]:
duplicates2.show()

In [0]:
spark.sql("""
  SELECT *
  FROM finance_project.loans_defaulters_delinq_external
  WHERE member_id LIKE 'e4c167053d5418230%'
""").show(10, False)

In [0]:
duplicates3 = spark.sql("""
  SELECT member_id, COUNT(*) AS total
  FROM finance_project.loans_defaulters_detail_records_enq_external
  GROUP BY member_id
  HAVING COUNT(*) > 1
  ORDER BY total DESC
""")

In [0]:
duplicates3.show()

In [0]:
spark.sql("""
  SELECT *
  FROM finance_project.loans_defaulters_detail_records_enq_external
  WHERE member_id LIKE 'e4c167053d5418230%'
""").show(10, False)

In [0]:
bad_data_customer_df = spark.sql("select member_id from (select member_id, count(*) as total from finance_project.customers_external group by member_id having total > 1) ")

In [0]:
bad_data_loans_defaulters_delinq_df = spark.sql("select member_id from (select member_id, count(*) as total from finance_project.loans_defaulters_delinq_external group by member_id having total > 1) ")

In [0]:
bad_data_customer_df.count()


In [0]:
bad_data_loans_defaulters_detail_records_enq_df =  spark.sql("select member_id from (select member_id, count(*) as total from finance_project.loans_defaulters_detail_records_enq_external group by member_id having total > 1) ")

In [0]:
bad_data_loans_defaulters_detail_records_enq_df.count()

In [0]:
mkdir -p "Volumes/finance_domain_project/default/raw_data/BadData"


In [0]:
bad_data_customer_df\
  .repartition(1)\
  .write\
  .option("header", "true")\
  .format("csv")\
  .mode("overwrite")\
  .option("path", "/Volumes/finance_domain_project/default/raw_data/BadData/bad_data_customers")\
  .save()

In [0]:
bad_data_loans_defaulters_delinq_df.repartition(1)\
  .write\
  .option("header", "true")\
  .format("csv")\
  .mode("overwrite")\
  .option("path", "/Volumes/finance_domain_project/default/raw_data/BadData/bad_data_loan_defaulters")\
  .save()

In [0]:
bad_data_loans_defaulters_detail_records_enq_df.repartition(1)\
  .write\
  .option("header", "true")\
  .format("csv")\
  .mode("overwrite")\
  .option("path", "/Volumes/finance_domain_project/default/raw_data/BadData/bad_data_loan_defaulters_detail_rec_enq")\
  .save()

In [0]:
bad_customer_data_df = bad_data_customer_df.select("member_id") \
    .union(bad_data_loans_defaulters_delinq_df.select("member_id")) \
    .union(bad_data_loans_defaulters_detail_records_enq_df.select("member_id"))


In [0]:
bad_customer_final_df = bad_customer_data_df.distinct()

In [0]:
bad_customer_final_df .repartition(1)\
  .write\
  .option("header", "true")\
  .format("csv")\
  .mode("overwrite")\
  .option("path", "/Volumes/finance_domain_project/default/raw_data/BadData/bad_customer_data_final")\
  .save()


In [0]:
bad_customer_final_df.createOrReplaceTempView("bad_data_customer")


In [0]:
customers_df = spark.sql("select * from finance_project.customers_external where member_id not in (select member_id from bad_data_customer)")


In [0]:
dbutils.fs.mkdirs("/Volumes/finance_domain_project/default/raw_data/newCleanedData")


In [0]:
customers_df.write \
  .format("parquet") \
  .mode("overwrite") \
  .option("path", "/Volumes/finance_domain_project/default/raw_data/newCleanedData/customers_parquet") \
  .save()

In [0]:
loans_defaulters_delinq_df = spark.sql("select * from finance_project.loans_defaulters_delinq_external where member_id not in (select member_id from bad_data_customer)")

In [0]:
loans_defaulters_delinq_df \
  .write \
  .format("parquet") \
  .mode("overwrite") \
  .option("path", "/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet") \
  .save()

In [0]:
loans_defaulters_detail_records_enq_df = spark.sql("select * from finance_project.loans_defaulters_detail_records_enq_external where member_id not in (select member_id from bad_data_customer)")


In [0]:
loans_defaulters_detail_records_enq_df \
  .write \
  .format("parquet") \
  .mode("overwrite") \
  .option("path", "/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet") \
  .save()

In [0]:
%sql
-- As owner, extend to cover entire container
ALTER EXTERNAL LOCATION finance_raw_location 
SET URL 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/';


In [0]:
%sql DESCRIBE EXTERNAL LOCATION finance_raw_location


In [0]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS finance_project.customers_new (
  member_id string, emp_title string, emp_length int, home_ownership string,
  annual_income float, address_state string, address_zip_code string, 
  address_country string, grade string, sub_grade string, 
  verification_status string, tot_hi_cred_limit float, 
  application_type string, join_annual_income float, 
  verification_status_joint string, ingest_date timestamp
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet'
""")


In [0]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS finance_project.loans_defaulters_delinq_new(member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet'
""")

In [0]:
%sql
DESCRIBE TABLE EXTENDED finance_project.customers_new;

In [0]:
%sql
SELECT * FROM finance_domain_project.finance_project.customers_new;

In [0]:
%sql
SELECT * FROM parquet.`dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/customers_parquet` LIMIT 29

In [0]:
spark.sql("""
CREATE EXTERNAL TABLE IF NOT EXISTS finance_project.loans_defaulters_detail_rec_enq_new(member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet'
""")

In [0]:
result = spark.sql("""select member_id, count(*) as total 
                      from finance_project.customers_new
                      group by member_id order by total desc""")

result.show()

In [0]:
%sql
DROP TABLE IF EXISTS finance_project.customers_new;


In [0]:
%sql
CREATE EXTERNAL TABLE finance_project.customers_new
(
  member_id string,
  emp_title string,
  emp_length int,
  home_ownership string,
  annual_income float,
  address_state string,
  address_zip_code string,
  address_country string,
  grade string,
  sub_grade string,
  verification_status string,
  tot_hi_cred_limit float,
  application_type string,
  join_annual_income float,
  verification_status_joint string,
  ingest_date timestamp
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7666432073935850>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE EXTERNAL TABLE finance_project.customers_new\n(\n  member_id string,\n  emp_title string,\n  emp_length int,\n  home_ownership string,\n  annual_income float,\n  address_state string,\n  address_zip_code string,\n  address_country string,\n  grade string,\n  sub_grade string,\n  verification_status string,\n  tot_hi_cred_limit float,\n  application_type string,\n  join_annual_income float,\n  verification_status_joint string,\n  ingest_date timestamp\n)\nUSING parquet\nLOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magi

In [0]:
spark.sql("SELECT * FROM finance_project.customers_new LIMIT 10").show()



In [0]:
spark.read.parquet("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet").show(5)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7666432073935848>, line 1
----> 1 spark.read.parquet("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet").show(5)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1156, in DataFrame.show(self, n, truncate, vertical)
   1155 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1156     print(self._show_string(n, truncate, vertical))

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:909, in DataFrame._show_string(self, n, truncate, vertical)
    892     except ValueError:
    893         raise PySparkTypeError(
    894             errorClass="NOT_BOOL",
    895             messageParameters={
   (...)
    898             

In [0]:
%sql DESCRIBE EXTERNAL LOCATION finance_raw_location;


In [0]:
dbutils.fs.ls("dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData")

In [0]:
%sql
DROP TABLE finance_project.customers_new;


In [0]:
%sql
CREATE EXTERNAL TABLE finance_project.customers_new
(
  member_id string,
  emp_title string,
  emp_length int,
  home_ownership string,
  annual_income float,
  address_state string,
  address_zip_code string,
  address_country string,
  grade string,
  sub_grade string,
  verification_status string,
  tot_hi_cred_limit float,
  application_type string,
  join_annual_income float,
  verification_status_joint string,
  ingest_date timestamp
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';

In [0]:
%sql
SELECT * FROM finance_project.customers_new LIMIT 10;


In [0]:
%sql DESCRIBE TABLE finance_project.customers_new;


In [0]:
%sql
SELECT current_catalog();

In [0]:
%sql
SHOW SCHEMAS;

In [0]:
%sql
SHOW TABLES IN finance_project;

In [0]:
customers_df.printSchema()

In [0]:
%sql
DROP TABLE finance_domain_project.finance_project.customers_new;

In [0]:
%sql
SHOW TABLES IN finance_project;

In [0]:
%sql
CREATE EXTERNAL TABLE finance_domain_project.finance_project.customers_new
(
  member_id string,
  emp_title string,
  emp_length int,
  home_ownership string,
  annual_income float,
  address_state string,
  address_zip_code string,
  address_country string,
  grade string,
  sub_grade string,
  verification_status string,
  tot_hi_cred_limit float,
  application_type string,
  join_annual_income float,
  verification_status_joint string,
  ingest_date timestamp
)
USING parquet
LOCATION 'dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6458695125425640>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "CREATE EXTERNAL TABLE finance_domain_project.finance_project.customers_new\n(\n  member_id string,\n  emp_title string,\n  emp_length int,\n  home_ownership string,\n  annual_income float,\n  address_state string,\n  address_zip_code string,\n  address_country string,\n  grade string,\n  sub_grade string,\n  verification_status string,\n  tot_hi_cred_limit float,\n  application_type string,\n  join_annual_income float,\n  verification_status_joint string,\n  ingest_date timestamp\n)\nUSING parquet\nLOCATION 'dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, 

In [0]:
dbutils.fs.cp(
    "dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/customers_parquet",
    "abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet",
    recurse=True
)

In [0]:
spark.read.parquet("abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet").show(5)

In [0]:
%sql
DROP TABLE finance_domain_project.finance_project.customers_new;



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6458695125425643>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DROP TABLE finance_domain_project.finance_project.customers_new;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:205, in SqlMagic.sql(self, line, cell)
    198 except BaseException as e:
    199     s

In [0]:
%sql
CREATE EXTERNAL TABLE finance_domain_project.finance_project.customers_new
(
  member_id string,
  emp_title string,
  emp_length int,
  home_ownership string,
  annual_income float,
  address_state string,
  address_zip_code string,
  address_country string,
  grade string,
  sub_grade string,
  verification_status string,
  tot_hi_cred_limit float,
  application_type string,
  join_annual_income float,
  verification_status_joint string,
  ingest_date timestamp
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/customers_parquet';

In [0]:

%sql
SELECT * FROM finance_domain_project.finance_project.customers_new LIMIT 10;

In [0]:
dbutils.fs.cp(
    "dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet",
    "abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet",
    recurse=True
)

In [0]:
%sql
DROP TABLE IF EXISTS finance_project.loans_defaulters_delinq_new;

In [0]:
loans_df = spark.read.parquet("dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet")
loans_df.printSchema()
loans_df.show(2, truncate=False) 

In [0]:
%sql
CREATE EXTERNAL TABLE finance_project.loans_defaulters_delinq_new (
  member_id string,
  delinq_2yrs int,
  delinq_amnt float,
  mths_since_last_delinq int
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_delinq_parquet';

In [0]:
%sql
SELECT COUNT(*) as row_count FROM finance_project.loans_defaulters_delinq_new;
SELECT * FROM finance_project.loans_defaulters_delinq_new LIMIT 5;

In [0]:
enq_df = spark.read.parquet("dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet")
enq_df.printSchema()
enq_df.show(2, truncate=False)

In [0]:
%sql
DROP TABLE IF EXISTS finance_project.loans_defaulters_detail_rec_enq_new;

In [0]:
dbutils.fs.cp(
    "dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet",
    "abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet",
    recurse=True
)


In [0]:
enq_df = spark.read.parquet("dbfs:/Volumes/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet")
enq_df.printSchema()

In [0]:
%sql
DROP TABLE IF EXISTS finance_project.loans_defaulters_detail_rec_enq_new;

In [0]:
%sql
CREATE EXTERNAL TABLE finance_project.loans_defaulters_detail_rec_enq_new (
  member_id string,
  pub_rec int,
  pub_rec_bankruptcies int,
  inq_last_6mths int
)
USING parquet
LOCATION 'abfss://raw-data@financeprojectdatalake01.dfs.core.windows.net/finance_domain_project/default/raw_data/newCleanedData/loans_defaulters_detail_records_enq_parquet';

In [0]:
%sql
SELECT member_id, COUNT(*) as total
FROM finance_project.customers_new 
GROUP BY member_id 
HAVING COUNT(*) > 1
ORDER BY total DESC;

In [0]:
%sql
SELECT member_id, COUNT(*) as total
FROM finance_project.loans_defaulters_delinq_new 
GROUP BY member_id 
HAVING COUNT(*) > 1
ORDER BY total DESC;

In [0]:
%sql
SELECT member_id, COUNT(*) as total
FROM finance_project.loans_defaulters_detail_rec_enq_new 
GROUP BY member_id 
HAVING COUNT(*) > 1
ORDER BY total DESC;